# AWQ diagnostic (Kaggle T4 ×1)

Hypothesis (unconfirmed): AWQ dequant→fp16 before GEMM loses at c=1 short decode.

**Rules:** `hardware=kaggle_t4_x1` only. Append CSV rows with `AWQ_DIAG` in notes. Do not delete prior ladder rows. Do not log laptop timings.

See `docs/awq_diag.md` for what patterns mean.

## 1) Clone / install

If the repo is already uploaded to `/kaggle/working/llm-serving-bench`, skip clone.

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("llm-serving-bench"):
    subprocess.check_call(["git", "clone", "https://github.com/kartikshelar/llm-serving-bench.git"])
os.chdir("/kaggle/working/llm-serving-bench")
print("cwd", os.getcwd())
# !pip install -q -r requirements.txt -r requirements-bench.txt

## 2) Run full diagnostic protocol

Starts/stops vLLM for fp16 and AWQ at `max_tokens` 64 / 256 / 1024, c=1, 3 repeats each, with GPU sampling.

In [ ]:
import subprocess
subprocess.check_call(["bash", "scripts/kaggle_awq_diag.sh"])

## 3) Quick preview of new rows (`errors==0`, `AWQ_DIAG`)

In [ ]:
import csv
from collections import defaultdict
from statistics import mean, stdev

rows = list(csv.DictReader(open("results/benchmarks.csv", newline="", encoding="utf-8")))
diag = [r for r in rows if "AWQ_DIAG" in (r.get("notes") or "") and int(float(r["errors"] or 0)) == 0]
print("AWQ_DIAG ok rows:", len(diag))
G = defaultdict(list)
for r in diag:
    note = r["notes"]
    mt = next((p.split("=")[1].rstrip(";") for p in note.split() if p.startswith("max_tokens=")), "?")
    key = (r["rung"], mt)
    G[key].append(float(r["output_tokens_per_sec"]))
for k in sorted(G, key=lambda x: (int(x[0]), int(x[1]) if str(x[1]).isdigit() else 0)):
    xs = G[k]
    sd = stdev(xs) if len(xs) > 1 else 0.0
    print(f"rung={k[0]} max_tokens={k[1]} n={len(xs)} mean={mean(xs):.1f} std={sd:.1f} min={min(xs):.1f} max={max(xs):.1f}")
print("\nDownload results/benchmarks.csv + results/awq_kernel_inspect.txt when done.")